# Muon's Newton-Schulz Orthogonalization

AI & ML Foundations: Optimizers · *Mathematics for AI and Machine Learning* by Fuheng Wu

Q <- Q(3I - Q^T Q)/2 drives every singular value to 1, giving the polar factor Muon uses as its update (Chapters 8, 11).

Companion code for the [interactive demo](https://math4ai.org/companion/ml/muon-newton-schulz/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(8)

# The demo's defaults: an 8 x 8 update G with condition number 50, ||Q0||_2 = 1, 12 iterations
n, kappa, iters = 8, 50.0, 12
U, _ = torch.linalg.qr(torch.randn(n, n, dtype=torch.float64))
V, _ = torch.linalg.qr(torch.randn(n, n, dtype=torch.float64))
s = 3 * kappa ** (-torch.arange(n, dtype=torch.float64) / (n - 1))       # from 3 down to 3/kappa
G = U @ torch.diag(s) @ V.T
polar = U @ V.T                                                         # orthogonal factor of G = Q S


def newton_schulz(G, iters, scale=1.0):
    Q = scale * G / torch.linalg.matrix_norm(G, ord=2)                  # ||Q0||_2 = scale
    I = torch.eye(G.shape[0], dtype=G.dtype)
    for _ in range(iters):
        Q = 0.5 * Q @ (3 * I - Q.T @ Q)                                # only matrix products
    return Q


print(" iters   singular values of Q (min, max)   ||Q - U V^T||")
for t in (0, 2, 4, 8, 12, 16):
    Q = newton_schulz(G, t)
    sv = torch.linalg.svdvals(Q)
    print(f"{t:5d}    ({sv.min().item():.6f}, {sv.max().item():.6f})        {torch.linalg.matrix_norm(Q - polar).item():.2e}")

# Each singular value follows the scalar map s -> s (3 - s^2) / 2, which pulls (0, sqrt 3) to 1
g = lambda v: v * (3 - v * v) / 2
print("\nscalar map from 0.02:", [round(v, 4) for v in [0.02, g(0.02), g(g(0.02)), g(g(g(0.02)))]], "...")

# ── Starting above sqrt(3) flips signs: orthogonal, but not the polar factor ─
Q = newton_schulz(G, 40, scale=1.9)
print(f"\nstart ||Q0|| = 1.9: ||Q^T Q - I|| = {torch.linalg.matrix_norm(Q.T @ Q - torch.eye(n, dtype=Q.dtype)).item():.1e}, "
      f"||Q - U V^T|| = {torch.linalg.matrix_norm(Q - polar).item():.3f}")

# Muon applies this orthogonal factor as the update: every direction gets step size 1
print("spectral norm of the Muon update:", torch.linalg.matrix_norm(newton_schulz(G, 30), ord=2).item())

 iters   singular values of Q (min, max)   ||Q - U V^T||
    0    (0.020000, 1.000000)        2.21e+00
    2    (0.044981, 1.000000)        1.89e+00
    4    (0.100985, 1.000000)        1.51e+00
    8    (0.478915, 1.000000)        5.86e-01
   12    (0.998441, 1.000000)        1.56e-03
   16    (1.000000, 1.000000)        1.10e-15

scalar map from 0.02: [0.02, 0.03, 0.045, 0.0674] ...

start ||Q0|| = 1.9: ||Q^T Q - I|| = 3.3e-16, ||Q - U V^T|| = 2.000
spectral norm of the Muon update: 1.0000000000000004
